# SABRE: Simulation of the coherent dynamics - 4 spin system
Before studying this notebook, it is recommended to go through the following:
- sabre_3-fp_3spin_simulation_cosine.ipynb

This notebook contains a spin dynamics simulation of a four-spin 3,5-difluoropyridine SABRE system. It demonstrates that the magnetic field parameters predicted by the generalised level anticrossings (LACs) derived using a three-spin system lead to efficient hyperpolarisation of $^{19}\mathrm{F}$ nuclei, convincing that the theory also holds for systems with multiple heteronuclei of the same type. In the simulation, the system is subjected to a sinusoidal polarisation transfer field $B_0 + B_1 \cos(\omega_p t)$, where $B_0$ is the average offset, $B_1$ the amplitude, and $\omega_p = 2\pi / T$ the angular frequency corresponding to the period $T$. The period $T$ is chosen to be $10\ \mathrm{ms}$, and the other parameters such that $B_0$ fulfills the generalised LAC condition with $m=1$ and that $B_1$ maximises effective coupling.

## Details
- Spin system: Four spins (hydride protons + two $\mathrm{^{19}F}$ nuclei from equatorial 3,5-difluoropyridine ligand)
- Only coherent dynamics are simulated (no chemical exchange or relaxation)
- Only the polarisation of each nucleus is plotted

The simulations are performed using Spinguin (v. 0.0.1) package. In addition, we use the functionality from NumPy and Matplotlib for creating arrays and plotting, respectively.

In [ ]:
import spinguin as sg
from spinguin.core.hide_prints import HidePrints
import numpy as np
import matplotlib.pyplot as plt

### Build the required spin system
We build the four-spin system that consists of the hydride protons and the two $^{19}\mathrm{F}$ nuclei from the 3,5-difluoropyridine ligand.

In [ ]:
ss_c = sg.SpinSystem(["1H", "1H", "19F", "19F"])
ss_c.basis.max_spin_order = 4
ss_c.basis.build()

### Define chemical shifts and $J$-couplings
We set the chemical shifts in ppm and the $J$-couplings in Hz.

In [ ]:
ss_c.chemical_shifts = [-23.2, -23.2, -124.4, -124.4]

ss_c.J_couplings = [
    [0,     0,      0,      0],
    [-7.01, 0,      0,      0],
    [-0.03, 2.44,   0,      0],
    [0.04,  2.24,   3.09,   0]
]

### Create the time evolution propagator
Next, the parameters for the sinusoidal polarisation transfer field with the period $T = 10\ \mathrm{ms}$ are defined. We set the average offset $B_0 = 43.1\ \mathrm{\mu T}$ (matches the generalised LAC condition) and $B_1 = 73.2\ \mathrm{\mu T}$ (maximises the effective coupling). These values were calculated in one of the previous notebooks. We approximate the sinusoidal magnetic field with $nterms = 100$ data points. For convenience, the time step of the propagator is chosen to be equal to one period of the oscillating field. We build the Hamiltonian, Liouvillian, and the time propagator. The time propagator corresponds now to one period of the wave.

In [ ]:
T = 10e-3
B0 = 43.1e-6
B1 = 73.2e-6

nterms = 100
dt = T / nterms

Bt = B0 + B1 * np.array([np.cos(i*2*np.pi / nterms) for i in range(nterms)])

P = np.eye(ss_c.basis.dim, ss_c.basis.dim)
with HidePrints():
    for i in range(nterms):
        sg.parameters.magnetic_field = Bt[i]
        H = sg.hamiltonian(ss_c)
        L = sg.liouvillian(H)
        P = sg.propagator(L, dt) @ P

### Time evolution
First, the spin singlet state is created for the hydride protons as the initial state. Then, we simulate the spin system for a short time of $1\ \mathrm{s}$ just to illustrate the coherent oscillation. The polarisations, which are acquired during each time step, are stored in an array for further analysis.

In [ ]:
rho_c = sg.singlet_state(ss_c, 0, 1)

time = 1
nsteps = int(time/T)
polarisations = np.zeros((nsteps, ss_c.nspins))

for i in range(nsteps):

    for j in range(ss_c.nspins):
        magnetisation = sg.measure(ss_c, rho_c, f"I(z,{j})").real
        polarisations[i, j] = magnetisation / ss_c.spins[j] * 100

    rho_c = P @ rho_c

### Plot the polarisations
The coherent evolution of the polarisation of each nucleus is plotted. It is observed that the polarisation of both $^{19}\mathrm{F}$ nuclei start to coherently oscillate. This is similar to the previous notebook, where the three-spin 3-fluoropyridine system was simulated. However, the maximum achievable polarisation is decreased. This is mainly due to now having two nuclei to-be-polarised, which halves the maximum polarisation, but also due to the fact that the parameters for the magnetic field were calculated using 3-fluoropyridine instead of 3,5-difluoropyridine. In addition, small quantitative differences in the magnetic field parameters are expected when including more spins in the system.

In [ ]:
t = np.linspace(0, nsteps*T, nsteps, endpoint=False)
for i in range(ss_c.nspins):
    plt.plot(t, np.real(polarisations[:, i]), label=f"Spin {i+1}: {ss_c.isotopes[i]}")
plt.legend(loc="upper right")
plt.xlabel("Time (s)")
plt.ylabel("Polarisation (%)")
plt.tight_layout()
plt.show()
plt.clf()

### Save the polarisations
Finally, the polarisations are saved as a .csv file for further analysis.

In [ ]:
np.savetxt(f"polarisations.csv", polarisations, delimiter=",")